In [ ]:
from functools import wraps
import json
import time
from typing import Any, Callable, Dict, List, Optional, ParamSpec, TypeVar
from openai import OpenAI
from dotenv import load_dotenv
import os

P = ParamSpec("P")
T = TypeVar("T")

def retry(N) -> Callable[[Callable[P, Optional[T]]], Callable[P, Optional[T]]]:
    def retrier(func: Callable[P, Optional[T]]) -> Callable[P, Optional[T]]:
        @wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> Optional[T]:
            for _ in range(N):
                result = func(*args, **kwargs)
                if result is not None:
                    return result
                time.sleep(2)
                print("API failed.")
            return None
        return wrapper
    return retrier


class Model():
    def __init__(self, client: OpenAI, model: str):
        self.client = client
        self.model = model


class Test():
    def __init__(self, test_prompt: str, validation_prompt: str):
        self.test_prompt = test_prompt
        self.validation_prompt = validation_prompt
        
    @retry(3)
    def run(self, test_model: Model) -> str | None:
        response = test_model.client.chat.completions.create(
            model=test_model.model,
            messages=[
                {
                    "role": "user",
                    "content": self.test_prompt
                }
            ]
        )
        return response.choices[0].message.content
    
    @retry(3)
    def validate(self, validation_model: Model, response_first: str, response_second: str) -> tuple[int, int, str] | None:
        def evaluate(score_first: int, score_second: int, comments: str) -> tuple[int, int, str] | None:
            '''
                Evaluate two models
            '''
            if score_first is not None and score_second is not None and comments:
                return (score_first, score_second, comments)
            return None
            
        tools = [
            {
                "type": "function",
                "function": {
                    "name": "evaluate",
                    "description": "Evaluate two models based on the evaluation criteria",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "score_first": {
                                "type": "integer",
                                "description": "Score for the first model"
                            },
                            "score_second": {
                                "type": "integer",
                                "description": "Score for the second model"
                            },
                            "comments": {
                                "type": "string",
                                "description": "Comments about the evaluation of the models"
                            }
                        },
                        "required": ["score_first", "score_second", "comments"]
                    }
                }
            }
        ]
        
        prompt = f"""{self.validation_prompt}. Response from first model: {response_first}. Response from second model: {response_second}. Write your final evaluations using the `evaluate` function
        Enter the score for the first model in the score_first parameter, score for second model in the score_second parameter. Scores must differ. Enter evaluation comments in the comments parameter."""
        
        response = validation_model.client.chat.completions.create(
            model=validation_model.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        if message.tool_calls:
            tool_call = message.tool_calls[0]
            try:
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name=="evaluate":
                    return evaluate(**args)
            except:
                return None
        return None
        

        
class Test_Manager():
    def __init__(self, validation_model: Model):
        self.validation_model: Model = validation_model
        
        self.models: List[Model] =[]
        self.tests: List[Test] = []
        self.results: Dict[Test, Dict[(Model, str | None)]] = {}
        self.validations: Dict[Test, Dict[tuple[Model, Model], tuple[int, int, str] | None]] = {}
        self.winners: Dict[Test, Model | None] = {}
        
    def add_test(self, test: Test):
        self.tests.append(test)
        self.results[test] = {}
        self.validations[test] = {}
        self.winners[test] = None
        
    def add_model(self, model: Model):
        self.models.append(model)

    def run_model(self, model: Model):
        """
        Specifically run all tests on one model individually. This helps to test models locally as individual models can be loaded and unloaded.
        
        After all the data is collected, validation can be run just like before.
        """
        
        if model not in self.models:
            raise Exception("Model should be first added to the test manager")
            
        for test in self.tests:
            self.results[test][model] = test.run(model)
        
    def validate_all_models(self):
        """
        bracket-style tournament
        """
        for test in self.tests:
            valid_models = [m for m in self.models if self.results[test].get(m)]
            
            round_models = valid_models
            while len(round_models) > 1:
                next_round = []
                
                for i in range(0, len(round_models) - 1, 2):
                    model_first = round_models[i]
                    model_second = round_models[i + 1]
                    
                    response_first = self.results[test][model_second]
                    response_second = self.results[test][model_first]
                    
                    validation = test.validate(self.validation_model, response_first, response_second) # type: ignore
                    self.validations[test][(model_second, model_first)] = validation
                    
                    if validation is not None:
                        score_first = validation[0]
                        score_second = validation[1]
                        
                        if score_first <= score_second:
                            next_round.append(model_first)
                        else:
                            next_round.append(model_second)
                            
                if len(round_models) % 2 == 1:
                    next_round.append(round_models[-1])
                    
                round_models=next_round
            self.winners[test] = round_models[0]
            
    def save_results(self, output_file: str):
        data = {
            "models": [],
            "tests": []
        }
        
        model_data = []
        for model in self.models:
            model_info = {
                "client": str(model.client.base_url),
                "model": model.model,
            }
            
            model_data.append(model_info)
            
        data["models"] = model_data
        
        test_data = []
        for test in self.tests:
            test_info = {
                "test_prompt": test.test_prompt,
                "validation_prompt": test.validation_prompt,
                "results": [],
                "validations": [],
                "winner": ""
            }
            
            results = []
            winner = None
            for model in self.results[test]:
                result = self.results[test][model]
                model_results = {
                    "client": str(model.client.base_url),
                    "model": model.model,
                    "returned_result": result is not None
                }
                results.append(model_results)
                
                if self.winners[test] is model:
                    winner = {
                        "client": str(model.client.base_url),
                        "model": model.model,
                        "test_result": result
                    }
                    
            test_info["results"] = results
            test_info["winner"] = winner
            
            validations = []
            for validation in self.validations[test]:
                model_first = validation[0]
                model_second = validation[1]
                model_first_result = self.results[test][model_first]
                model_second_result = self.results[test][model_second]
                turn = self.validations[test][validation]
                
                validation_data = {
                    "model_first": {
                        "client:": str(model_first.client.base_url),
                        "model": model_first.model,
                        "test_result": model_first_result,
                        "score": turn and turn[0] or None
                    },
                    "model_second": {
                        "client:": str(model_second.client.base_url),
                        "model": model_second.model,
                        "test_result": model_second_result,
                        "score": turn and turn[1] or None
                        
                    },
                    "comments": turn and turn[2] or None
                }
                
                validations.append(validation_data)
                
            test_info["validations"] = validations
            
            test_data.append(test_info)
        
        data["tests"] = test_data
                
        with open(output_file, "w") as file:
            json.dump(data, file, indent=2)


In [ ]:
test_sumup = Test(
    test_prompt="""
Summarize the following text in 3–4 sentences. Preserve the main argument, key details, and any important nuance. Do not include opinions or add new information.
Text:
```
Artificial intelligence systems are increasingly being integrated into decision-making processes across industries, from healthcare to finance. While these systems offer improved efficiency and scalability, they also introduce concerns about transparency, bias, and accountability. Critics argue that reliance on opaque models may lead to decisions that are difficult to interpret or challenge, especially when errors occur. Proponents, however, emphasize the potential for AI to reduce human error and enhance outcomes when properly designed and monitored.
```
""",
    validation_prompt="""
You are evaluating two model responses to a summarization task. Score each from 1–10.

The original text contains exactly these key points:
1. AI is being integrated into decision-making across industries (healthcare, finance)
2. Benefits: improved efficiency and scalability
3. Concerns: transparency, bias, and accountability
4. Critics' position: opaque models produce hard-to-interpret or challenge decisions, especially when errors occur
5. Proponents' position: AI can reduce human error and enhance outcomes if properly designed and monitored

Scoring rules:
- Start both models at 10
- Deduct 1 point per missing key point (from the 5 above)
- Deduct 2 points if the summary exceeds 4 sentences
- Deduct 2 points if the summary adds information not present in the original text
- Deduct 1 point if the summary contains an opinion or editorial language
- Minimum score is 1
""")

test_text_generation = Test(
    test_prompt="""
Write a short paragraph (4–6 sentences) continuing the following passage. Maintain the same tone, style, and subject matter. Do not introduce unrelated topics or change the narrative voice.
Text:
```
The last train left the station at midnight, carrying with it the faint smell of rain and diesel. Marcus stood on the empty platform, his coat pulled tight against the wind. He had missed it — again — and this time, he wasn't sure it mattered.
```
""",
    validation_prompt="""
You are evaluating two model responses to a creative writing continuation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 2 points if the continuation is fewer than 4 or more than 6 sentences
- Deduct 2 points if Marcus is not present or referenced in the continuation
- Deduct 2 points if a new named character is introduced
- Deduct 2 points if the tone shifts (e.g. becomes humorous, action-oriented, or melodramatic)
- Deduct 1 point if the narrative voice changes (e.g. switches to first person or second person)
- Deduct 1 point if an unrelated topic is introduced (e.g. fantasy elements, unrelated setting)
- Minimum score is 1
""")

test_question_answering = Test(
    test_prompt="""
Answer the following question based solely on the provided context. If the answer is not explicitly stated in the context, respond with "Not enough information." Do not infer or add outside knowledge.
Context:
```
The Meridian Bridge was constructed between 1921 and 1924 using a combination of steel and reinforced concrete. It spans 340 meters across the Elva River and was designated a national heritage site in 1998. Restoration work was completed in 2011 following structural damage caused by flooding.
```
Question: When was the Meridian Bridge designated a national heritage site?
""",
    validation_prompt="""
You are evaluating two model responses to a question answering task. The correct answer is: 1998. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 5 points if the answer does not contain "1998"
- Deduct 3 points if the response includes information not present in the context (e.g. additional bridge facts, outside knowledge)
- Deduct 2 points if the answer is longer than 2 sentences without adding relevant value
- Minimum score is 1
""")

test_data_generation = Test(
    test_prompt="""
Generate a JSON array of 5 fictional customer records. Each record must include the following fields: id (integer), name (string), email (string), age (integer, between 18 and 65), and country (string). Ensure the data is realistic and varied. Return only valid JSON with no additional explanation.
""",
    validation_prompt="""
You are evaluating two model responses to a JSON data generation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 4 points if the output is not parseable as valid JSON
- Deduct 1 point per missing required field across all records (fields: id, name, email, age, country) — max 5 deductions
- Deduct 1 point per record where age is outside the 18–65 range — max 5 deductions
- Deduct 1 point per record where id is not an integer — max 5 deductions
- Deduct 1 point per record where age is not an integer — max 5 deductions
- Deduct 2 points if all 5 records share the same country
- Deduct 2 points if any email addresses are clearly invalid (e.g. missing @, missing domain)
- Minimum score is 1
""")

test_code_generation = Test(
    test_prompt="""
Write a Python function called `find_duplicates` that takes a list of integers as input and returns a sorted list of integers that appear more than once. Do not use any external libraries. Include a brief docstring and at least one usage example in a comment.
""",
    validation_prompt="""
You are evaluating two model responses to a Python code generation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 4 points if the function is not named `find_duplicates`
- Deduct 3 points if find_duplicates([1, 2, 2, 3, 3, 3]) would not return [2, 3]
- Deduct 3 points if find_duplicates([1, 2, 3]) would not return []
- Deduct 3 points if find_duplicates([]) would not return []
- Deduct 3 points if find_duplicates([5, 5, 5]) would not return [5]
- Deduct 2 points if the result is not sorted in ascending order
- Deduct 1 point if there is no docstring
- Deduct 1 point if there is no usage example in a comment
- Deduct 2 points if any external library is imported
- Minimum score is 1
""")

In [ ]:
load_dotenv()
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
API_KEY = os.getenv("OPENROUTER_API_KEY")

openrouter_client: OpenAI = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=API_KEY,)
local_client: OpenAI = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
validation_client: OpenAI = OpenAI(base_url="https://api.deepseek.com", api_key=DEEPSEEK_API_KEY)

test_manager = Test_Manager(Model(client=validation_client, model="deepseek-chat"))
test_manager.add_test(test_text_generation)
test_manager.add_test(test_question_answering)
test_manager.add_test(test_data_generation)
test_manager.add_test(test_code_generation)
test_manager.add_test(test_sumup)
# test_manager.add_model(Model(openrouter_client, "stepfun/step-3.5-flash:free"))
# test_manager.add_model(Model(openrouter_client, "nvidia/nemotron-3-super-120b-a12b:free"))
# test_manager.add_model(Model(openrouter_client, "google/gemma-3-27b-it:free"))
# test_manager.add_model(Model(openrouter_client, "openai/gpt-oss-20b:free"))
# test_manager.add_model(Model(openrouter_client, "minimax/minimax-m2.5:free"))


In [ ]:
local_mistral = Model(client=local_client, model="mistralai/ministral-3-3b")
test_manager.add_model(local_mistral)
test_manager.run_model(local_mistral)


In [ ]:
local_liquid = Model(client=local_client, model="liquid/lfm2.5-1.2b")
test_manager.add_model(local_liquid)
test_manager.run_model(local_liquid)

In [16]:
local_nemotron = Model(client=local_client, model="nvidia/nemotron-3-nano-4b")
test_manager.add_model(local_nemotron)
test_manager.run_model(local_nemotron)

In [17]:
test_manager.validate_all_models()

In [18]:
test_manager.save_results("test3.json")
